# 

`crisprVerse` env

> ### [dCas9 and other “dead” nucleases](https://github.com/crisprVerse/crisprBase?tab=readme-ov-file#dcas9-and-other-dead-nucleases)
The CRISPR inhibition (CRISPRi) and CRISPR activation (CRISPRa) technologies uses modified versions of CRISPR nucleases that lack endonuclease activity, often referred to as “dead Cas” nucleases, such as the dCas9.

> While fully-active Cas nucleases and dCas nucleases differ in terms of applications and type of genomic perturbations, <u>the gRNA design remains unchanged</u> in terms of spacer sequence search and genomic coordinates. Therefore it is convenient to use the fully-active version of the nuclease throughout crisprBase.

___
related links:
- https://github.com/crisprVerse/Tutorials/tree/master/Validating_Existing_gRNA_Libraries
- https://github.com/crisprVerse/Tutorials/tree/master/Building_Genome_Indices
- https://github.com/crisprVerse/crisprScore?tab=readme-ov-file#crisprai

In [209]:
library("GenomicRanges")

In [54]:
library(crisprVerse)
library(crisprScore)
library(tidyverse)

library(BSgenome.Hsapiens.UCSC.hg38)
bsgenome <- BSgenome.Hsapiens.UCSC.hg38

In [137]:
data(txdb_human, package="crisprDesignData")

txObject <- txdb_human

In [28]:
match_aln <-function(data,aln){
    # We add the genomic coordinates to the data.frame:

    wh <- match(data$sequence, aln$spacer)
    data$chr <- aln$chr[wh]
    data$pam_site <- aln$pam_site[wh]
    data$pam <- aln$pam[wh]
    data$strand <- aln$strand[wh]
    return (data)
}

make_gs <-function(data){
    gs <- GuideSet(ids=data$ID %>% as.list,
                   protospacers=data$sequence,
                   pams=data$pam,
                   pam_site=data$pam_site,
                   seqnames=data$chr,
                   strand=data$strand,
                   CrisprNuclease=SpCas9,
                   bsgenome = bsgenome)
    gs$gene_symbol <- data$gene
    
    return(gs)
}

#### 

In [3]:
library_path = '../ScreenPro2/library_tables/CRISPRi_v2_human_librarytable.txt.gz'

library = read.table(library_path,header=T)

In [106]:
library_filt = library %>% as.tibble %>% distinct(sequence,.keep_all = T)

filter duplicated sgRNAs.

In [117]:
library %>% nrow - library_filt %>% nrow

[1] 3414

___

In [121]:
# library %>% nrow

In [6]:
# data(package="crisprBase")

In [122]:
SpCas9

Class: CrisprNuclease
  Name: SpCas9
  Target type: DNA
  Metadata: list of length 1
  PAMs: NGG, NAG, NGA
  Weights: 1, 0.2593, 0.0694
  Spacer length: 20
  PAM side: 3prime
    Distance from PAM: 0
  Prototype protospacers: 5'--SSSSSSSSSSSSSSSSSSSS[NGG]--3', 5'--SSSSSSSSSSSSSSSSSSSS[NAG]--3', 5'--SSSSSSSSSSSSSSSSSSSS[NGA]--3'

In [123]:
spacer_sequences = library_filt$sequence %>% unique

In [124]:
spacer_sequences %>% length

[1] 205656

In [125]:
bowtie_index = '~/genomes/hg38/hg38'

In [126]:
aln <- runCrisprBowtie(
    spacers = spacer_sequences,
    bsgenome = bsgenome,
    crisprNuclease=SpCas9,
    bowtie_index=bowtie_index,
    n_mismatches=1
)

[runCrisprBowtie] Searching for SpCas9 protospacers 


In [127]:
chrs <- paste0("chr",c(1:22, "X", "Y"))
aln <- aln[aln$chr %in% chrs,,drop=FALSE]

In [130]:
library_aln = match_aln(library_filt,aln) %>% select(-sgId,-transcripts) %>%
  group_by(gene) %>%
  mutate(Series = 1:n()) %>%
  ungroup() %>%
  mutate(ID = paste0(gene, "_sg", Series),.before = sublibrary) %>%
  select(-Series) # Remove the intermediate series column

In [134]:
gs = make_gs(library_aln %>% drop_na)

### Off-target characterization

In [139]:
gs <- addSpacerAlignments(gs,
                          txObject=txObject,
                          aligner="bowtie",
                          aligner_index=bowtie_index,
                          bsgenome=bsgenome,
                          n_mismatches=2)

Loading required namespace: crisprBwa



[runCrisprBowtie] Using BSgenome.Hsapiens.UCSC.hg38 
[runCrisprBowtie] Searching for SpCas9 protospacers 


In [141]:
gs <- addOffTargetScores(gs)

Warning message in .local(object, ...):
"There are duplicated gRNAs in the GuideSet object, which can lead to incorrect off-target scores. Consider removing duplicated gRNAs first before running addOffTargetScores. "


In [143]:
# curl --cookie zenodo-cookies.txt \
# "https://zenodo.org/record/6716721/files/6716721.zip?download=1" --output 6716721.zip

> The scores range from 0 to 1, and a higher score indicates a higher probability of the off-target to occur.



> - n0, n1, n2, n3: specify number of alignments with 0, 1, 2 and 3 mismatches, respectively.
> - n0_c, n1_c, n2_c, n3_c: specify number of alignments in a coding region, with 0, 1, 2 and 3 mismatches, respectively.
> - n0_p, n1_p, n2_p, n3_p: specify number of alignments in a promoter region of a coding gene, with 0, 1, 2 and 3 mismatches, respectively.


#### on-target activity

> predicting on-target activity of the wildtype SpCas9 nuclease,
> - Rule Set
> - Azimuth algorithms (iteration of the popular Rule Set 2 algorithm by the same authors)
> - sequence-only Rule Set. 

> The package also provides the deep learning-based algorithms DeepWT and DeepHF, developed to predict cutting efficiency of the wildtype SpCas9 and SpCas9-High Fidelity (SpCas9-HF1) nucleases, respectively, and the DeepSpCas9 algorithm. 

> We also included the CRISPRscan algorithm for predicting on-target activity of SpCas9 gRNAs expressed from a T7 promoter, as well as the CRISPRater algorithm.

In [145]:
gs <- addOnTargetScores(gs, methods="crisprater")

[addOnTargetScores] Adding crisprater scores. 




___

In [152]:
gs <- addSequenceFeatures(gs)

In [159]:
# gs <- addGeneAnnotation(gs,
#                         anchor = c("pam_site"),
#                         txObject=txdb_human)

In [158]:
# geneAnnotation(gs)

___

In [157]:
txdb_human_tss = getTssObjectFromTxObject(txdb_human)

In [162]:
gs <- addTssAnnotation(gs,
                       anchor = c("pam_site"),
                       tssObject=txdb_human_tss)

In [201]:
# protospacer            pam
# pam_site gene_symbol
# n0        n1        n2      n0_c      n1_c      n2_c
# score_cfd score_mit
# score_crisprater percentGC     polyA     polyC     polyG     polyT
# startingGGGGG        NNGG

df = gs[,c(
    'protospacer','pam',
    'pam_site',
    'score_cfd','score_mit','score_crisprater'
    #'n0        n1        n2      n0_c      n1_c      n2_c
    # 'gene_symbol',
    # 'percentGC','polyA','polyC','polyG','polyT',
    # 'startingGGGGG','NNGG'
)] %>% data.frame %>% select(-width) %>% mutate(sgID=gs %>% names,.before='seqnames')

In [250]:
df %>% write.table('../ScreenPro2/library_tables/CRISPRi_v2_human_librarytable_crisprVerse.txt')

### 
add annotation

In [208]:
# txdb_human_tss

In [244]:
# ol = findOverlaps(df %>% GRanges, promoters(txdb_human$transcripts))

In [246]:
# distanceToNearest(df %>% GRanges, promoters(genes))

In [247]:
# genes <- rtracklayer::import('genes.bed')

In [248]:
# gene.obj = readTranscriptFeatures("../annotations/genes.bed")

### Save objects

In [252]:
saveRDS(aln,'rdata/aln.rds')

In [253]:
saveRDS(library_aln,'rdata/library_aln.rds')

In [254]:
saveRDS(gs,'rdata/gs.rds')

# 

In [255]:
sessionInfo()

R version 4.3.2 (2023-10-31)
Platform: x86_64-conda-linux-gnu (64-bit)
Running under: Debian GNU/Linux 11 (bullseye)

Matrix products: default
BLAS/LAPACK: /opt/conda/envs/crisprVerse/lib/libopenblasp-r0.3.25.so;  LAPACK version 3.11.0

locale:
 [1] LC_CTYPE=C.UTF-8       LC_NUMERIC=C           LC_TIME=C.UTF-8       
 [4] LC_COLLATE=C.UTF-8     LC_MONETARY=C.UTF-8    LC_MESSAGES=C.UTF-8   
 [7] LC_PAPER=C.UTF-8       LC_NAME=C              LC_ADDRESS=C          
[10] LC_TELEPHONE=C         LC_MEASUREMENT=C.UTF-8 LC_IDENTIFICATION=C   

time zone: Etc/UTC
tzcode source: system (glibc)

attached base packages:
[1] stats4    stats     graphics  grDevices utils     datasets  methods  
[8] base     

other attached packages:
 [1] BSgenome.Hsapiens.UCSC.hg38_1.4.5 BSgenome_1.70.1                  
 [3] rtracklayer_1.62.0                BiocIO_1.12.0                    
 [5] Biostrings_2.70.1                 XVector_0.42.0                   
 [7] GenomicRanges_1.54.1              GenomeInfoDb